# ValueFlow: Measuring Value Perturbation Propagation in Multi-Agent LLM Systems

Reproducing results from: [*ValueFlow* (arxiv 2602.08567)](https://arxiv.org/abs/2602.08567)

**Research question:** How do value perturbations propagate through multi-agent LLM systems,
and how are susceptibility and propagation dynamics shaped by interaction topology,
LLM backbone, perturbed value type, and perturbation location?

**Run pipeline before using this notebook:**
```bash
# 1. Simulate
uv run python scripts/run_valueflow.py --topologies chain ring star fully_connected
# 2. Notebook loads metrics computed automatically by run_valueflow.py
```

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

# Add repo root to path so scenario modules are importable
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scenarios.valueflow.plotting import (
    plot_beta_heatmap,
    plot_beta_timeseries,
    plot_location_effect,
    plot_ss_by_topology,
    plot_ss_by_value_type,
    plot_summary_grid,
)
from scenarios.valueflow.metrics import RunResults, compute_all_metrics, compute_cross_topology_comparison

# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 100,
    "axes.grid": False,
})

# ── Paths ─────────────────────────────────────────────────────────────────────
EXP_ROOT    = REPO_ROOT / "experiments" / "valueflow"
RESULTS_DIR = EXP_ROOT / "results"
STUDY_YAML  = EXP_ROOT / "study.yaml"

# ── Load study definition ─────────────────────────────────────────────────────
with open(STUDY_YAML) as f:
    study_def = yaml.safe_load(f)

study      = study_def["study"]
hypotheses = study_def["hypotheses"]

print(f"Study    : {study['name']}")
print(f"Question : {study['question']}")
print(f"Hypotheses: {list(hypotheses.keys())}")

In [ ]:
def load_metrics(path: str | Path) -> dict | None:
    """Load valueflow_metrics.json; return None if not yet produced."""
    p = REPO_ROOT / path
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)


def load_probe_results(path: str | Path) -> RunResults | None:
    """Load probe_results.jsonl; return None if not yet produced."""
    p = REPO_ROOT / path
    if not p.exists():
        return None
    return RunResults.from_jsonl(p)


# ── H1 topology results ───────────────────────────────────────────────────────
H1_CONDITIONS = ["chain", "ring", "star", "fully_connected"]
h1_metrics = {}
for topo in H1_CONDITIONS:
    m = load_metrics(f"experiments/valueflow/results/{topo}__social_power__agent0/valueflow_metrics.json")
    if m is not None:
        h1_metrics[topo] = m

# ── H3 value-type results ─────────────────────────────────────────────────────
H3_CONDITIONS = {
    "power":        "chain__social_power__agent0",
    "achievement":  "chain__ambitious__agent0",
    "universalism": "chain__equality__agent0",
    "benevolence":  "chain__helpful__agent0",
    "tradition":    "chain__respect_for_tradition__agent0",
    "security":     "chain__social_order__agent0",
}
h3_metrics = {}
for vtype, result_dir in H3_CONDITIONS.items():
    m = load_metrics(f"experiments/valueflow/results/{result_dir}/valueflow_metrics.json")
    if m is not None:
        h3_metrics[vtype] = m

# ── H4 location results ───────────────────────────────────────────────────────
H4_CONDITIONS = {0: "chain__social_power__agent0", 2: "chain__social_power__agent2", 4: "chain__social_power__agent4"}
h4_metrics = {}
for idx, result_dir in H4_CONDITIONS.items():
    m = load_metrics(f"experiments/valueflow/results/{result_dir}/valueflow_metrics.json")
    if m is not None:
        h4_metrics[idx] = m

# ── Baseline probe results (H5) ───────────────────────────────────────────────
baseline_results = load_probe_results("experiments/valueflow/results/baseline/probe_results.jsonl")

loaded = sum([
    len(h1_metrics), len(h3_metrics), len(h4_metrics),
    int(baseline_results is not None),
])
print(f"Loaded {loaded} result sets")
print(f"  H1 topologies : {list(h1_metrics.keys()) or '(no runs yet)'}")
print(f"  H3 value types: {list(h3_metrics.keys()) or '(no runs yet)'}")
print(f"  H4 locations  : {list(h4_metrics.keys()) or '(no runs yet)'}")
print(f"  Baseline       : {'loaded' if baseline_results else '(no runs yet)'}")

## Study Overview

ValueFlow measures how an amplified value in one agent propagates through a multi-agent network. The core idea:

1. **Baseline run** — all agents are neutral; probe all 56 Schwartz values per agent per round.
2. **Perturbed run** — one agent's persona is overridden to strongly endorse a target value; same probing.
3. **β-susceptibility** for non-perturbed agent $i$ on value $v$: $\beta_i(v) = s_i^{\text{pert}}(v) - s_i^{\text{base}}(v)$
4. **System Susceptibility (SS)**: $\text{SS}(v, \text{target}) = \text{mean}_{i \neq \text{target}}|\beta_i(v)|$

| Hypothesis | IV | Prediction |
|---|---|---|
| **H1** topology | topology type | SS: fully_connected > star > ring > chain |
| **H2** model backbone | LLM model | Larger/RLHF models show lower SS |
| **H3** value type | Schwartz value | SS: power > achievement > benevolence > tradition ≈ security |
| **H4** perturbation location | perturbed_agent_index | SS decreases with distance from head |
| **H5** baseline stability | perturbation_enabled | Without perturbation, scores stable (\|Δ\| < 0.5) |

In [ ]:
# Summarise what conditions have been run
print(f"{'Hypothesis':<28} {'Condition':<26} {'Status'}")
print("-" * 70)
for h_id, h_def in hypotheses.items():
    for cond_name in h_def.get("conditions", {}):
        run_path = None
        for run in h_def["conditions"][cond_name].get("runs", []):
            run_path = run.get("eval", "")
        p = REPO_ROOT / run_path if run_path and "TBD" not in run_path else None
        status = "✓ done" if (p and p.exists()) else "○ planned"
        print(f"{h_id:<28} {cond_name:<26} {status}")

## Key Metrics Explained

### 1. β-susceptibility (agent level)

For non-perturbed agent $i$, Schwartz value $v$:

$$\beta_i(v) = s_i^{\text{perturbed}}(v) \;-\; s_i^{\text{baseline}}(v)$$

where scores are SVS ratings on 0–10 collected by a probe at the end of the final round. Positive $\beta$ means the agent shifted *toward* the perturbed value; negative means it shifted away. Because we track all 56 Schwartz values simultaneously we can detect *spill-over*: the perturbation of one value shifting scores on unrelated values.

**Why it matters:** β-susceptibility gives agent-level resolution — we can identify which downstream agents were most affected and whether the effect was value-specific or diffuse.

---

### 2. System Susceptibility (SS)

The system-level headline metric. Aggregates β across all non-perturbed agents:

$$\text{SS}(v, \text{target}) = \frac{1}{|\mathcal{A} \setminus \{\text{target}\}|} \sum_{i \neq \text{target}} |\beta_i(v)|$$

Higher SS means the perturbation spread more strongly through the system. SS = 0 would mean no other agent was affected at all.

**Why it matters:** SS is the single number used for cross-condition comparison (H1, H2, H3, H4). It collapses the full β distribution into a scalar that quantifies how much a single adversarially-seeded agent can shift the whole network.

---

### 3. β-timeseries

β-susceptibility computed at each interaction round rather than only at the final step:

$$\beta_i^{(t)}(v) = s_i^{\text{pert}}(v,\, t) - s_i^{\text{base}}(v,\, t)$$

**Why it matters:** The timeseries reveals *propagation dynamics* — does influence accumulate gradually, spike at round 1, or reverse in later rounds?

---

### 4. Cross-value SS

SS computed on every Schwartz value (not just the perturbed one). Reveals whether perturbation of one value causes correlated shifts on conceptually related values (e.g. perturbing `social_power` also lifts `authority`).

**Why it matters:** The β-heatmap (agents × values) shows the full spillover pattern — the signature of how a value perturbation ripples across the entire Schwartz value space.

## H1: Topology Effect on System Susceptibility

**Statement:** Interaction topology significantly determines how a value perturbation propagates.
Fully-connected networks expose all agents to the perturbed node, so SS should be highest there;
chains attenuate influence with topological distance.

**Prediction:** SS(fully_connected) > SS(star) > SS(ring) > SS(chain)

In [ ]:
if h1_metrics:
    topology_comparison = {t: {"target_value_ss": m["target_value_ss"]} for t, m in h1_metrics.items()}
    fig, ax = plot_ss_by_topology(topology_comparison, target_value="social_power")
    plt.show()

    # Print ranked table
    ranked = sorted(topology_comparison.items(), key=lambda x: x[1]["target_value_ss"], reverse=True)
    print(f"\n{'Topology':<18} {'SS':>8}")
    print("-" * 28)
    for topo, vals in ranked:
        print(f"{topo:<18} {vals['target_value_ss']:>8.3f}")
else:
    print("No H1 results yet. Run: uv run python scripts/run_valueflow.py --topologies chain ring star fully_connected")

In [ ]:
# β-heatmap for the fully_connected condition (highest SS expected)
if "fully_connected" in h1_metrics:
    fc_metrics = h1_metrics["fully_connected"]

    # Build value_type_map from the probe result value_type field (if available)
    # Falls back to no grouping if types are missing
    value_type_map: dict[str, str] | None = None
    # If metrics contain value_type info, reconstruct map here

    fig, ax = plot_beta_heatmap(fc_metrics["beta_susceptibility"], value_type_map=value_type_map)
    ax.set_title("β-Susceptibility Heatmap — fully_connected topology\n(perturbed value: social_power)")
    plt.show()
elif h1_metrics:
    # Use whatever topology was run
    first_topo = list(h1_metrics.keys())[0]
    fig, ax = plot_beta_heatmap(h1_metrics[first_topo]["beta_susceptibility"])
    ax.set_title(f"β-Susceptibility Heatmap — {first_topo} topology")
    plt.show()
else:
    print("No H1 results yet.")

In [ ]:
# Propagation timeseries: social_power β per agent over rounds (chain topology)
if "chain" in h1_metrics:
    fig, ax = plot_beta_timeseries(
        h1_metrics["chain"]["beta_timeseries"],
        value_name="social_power",
    )
    ax.set_title("β-Susceptibility Over Rounds — chain topology (social_power)")
    plt.show()

    print("\nTimeseries shows whether influence accumulates gradually or spikes early.")
else:
    print("No chain topology results yet.")

## H3: Value Type Effect

**Statement:** Schwartz value types differ in how readily they propagate. Power and achievement values
(self-enhancing) spread more easily through a network than tradition or conformity values (conservation).

**Prediction:** SS(power) > SS(achievement) > SS(benevolence) > SS(tradition) ≈ SS(security)

*(Paper Fig. 4 — SS by value type grouped by Schwartz category)*

In [ ]:
if h3_metrics:
    ss_by_value_type = {vtype: m["target_value_ss"] for vtype, m in h3_metrics.items()}
    fig, ax = plot_ss_by_value_type(ss_by_value_type)
    plt.show()

    print(f"\n{'Value Type':<18} {'SS':>8}")
    print("-" * 28)
    for vtype, ss in sorted(ss_by_value_type.items(), key=lambda x: x[1], reverse=True):
        print(f"{vtype:<18} {ss:>8.3f}")
else:
    print("No H3 results yet. Run: uv run python scripts/run_valueflow.py --values social_power ambitious equality helpful respect_for_tradition social_order")

## H4: Perturbation Location in Chain

**Statement:** In a chain topology, perturbing an upstream agent (index 0) maximises propagation reach;
perturbing a downstream agent limits impact to a smaller reachable subgraph,
so SS decreases monotonically with position.

**Prediction:** SS(index=0) > SS(index=2) > SS(index=4) for a 5-agent chain.

*(Paper Fig. 6 — bar chart of SS by perturbation position)*

In [ ]:
if h4_metrics:
    ss_by_location = {idx: m["target_value_ss"] for idx, m in h4_metrics.items()}
    fig, ax = plot_location_effect(ss_by_location, n_agents=5)
    plt.show()

    print(f"\n{'Position':<10} {'Agent index':>12} {'SS':>8}")
    print("-" * 34)
    labels = {0: "head", 2: "middle", 4: "tail"}
    for idx in sorted(ss_by_location):
        label = labels.get(idx, f"index={idx}")
        print(f"{label:<10} {idx:>12} {ss_by_location[idx]:>8.3f}")

    if len(ss_by_location) >= 2:
        sorted_indices = sorted(ss_by_location)
        if ss_by_location[sorted_indices[0]] > ss_by_location[sorted_indices[-1]]:
            print("\n✓ Monotonic decrease with position: prediction supported.")
        else:
            print("\n✗ Non-monotonic: prediction not supported — investigate.")
else:
    print("No H4 results yet. Run: uv run python scripts/run_valueflow.py --locations 0 2 4")

## H2: Model Backbone Effect

**Statement:** Larger / RLHF-aligned models show lower susceptibility to value perturbation.

**Prediction:** SS(gpt-4o-mini) < SS(gpt-4o) across all conditions (smaller model = more susceptible)

Results directory for gpt-4o-mini: `experiments/valueflow/results_gpt4mini/`

In [ ]:
# ── Load gpt-4o-mini results ──────────────────────────────────────────────────
MINI_DIR = REPO_ROOT / "experiments" / "valueflow" / "results_gpt4mini"

def load_mini(label):
    p = MINI_DIR / label / "valueflow_metrics.json"
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

h1_mini = {t: load_mini(f"{t}__social_power__agent0") for t in H1_CONDITIONS}
h1_mini = {k: v for k, v in h1_mini.items() if v}

h3_mini = {vtype: load_mini(label) for vtype, label in H3_CONDITIONS.items()}
h3_mini = {k: v for k, v in h3_mini.items() if v}

h4_mini = {idx: load_mini(label) for idx, label in H4_CONDITIONS.items()}
h4_mini = {k: v for k, v in h4_mini.items() if v}

print(f"gpt-4o-mini loaded: H1={len(h1_mini)}/4  H3={len(h3_mini)}/6  H4={len(h4_mini)}/3")

# ── Side-by-side comparison: H1 topology ─────────────────────────────────────
models = {"gpt-4o": h1_metrics, "gpt-4o-mini": h1_mini}
topos = H1_CONDITIONS
x = np.arange(len(topos))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# H1
ax = axes[0]
for i, (model, mdict) in enumerate(models.items()):
    ss_vals = [mdict.get(t, {}).get("target_value_ss", float("nan")) for t in topos]
    bars = ax.bar(x + i * width, ss_vals, width, label=model,
                  color=["#e63946", "#f4a261"][i], alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, ss_vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(topos, rotation=15, ha="right", fontsize=8)
ax.set_ylabel("SS(social_power)")
ax.set_title("H1: Topology")
ax.legend(fontsize=8)
ax.set_ylim(0, max(3.5, ax.get_ylim()[1]))
ax.spines[["top", "right"]].set_visible(False)

# H3
ax = axes[1]
vtypes = list(H3_CONDITIONS.keys())
x3 = np.arange(len(vtypes))
for i, (model, mdict) in enumerate(models.items()):
    m3 = {"gpt-4o": h3_metrics, "gpt-4o-mini": h3_mini}[model]
    ss_vals = [m3.get(vt, {}).get("target_value_ss", float("nan")) for vt in vtypes]
    bars = ax.bar(x3 + i * width, ss_vals, width, label=model,
                  color=["#457b9d", "#a8dadc"][i], alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, ss_vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x3 + width / 2)
ax.set_xticklabels(vtypes, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("SS(target value)")
ax.set_title("H3: Value Type (chain)")
ax.legend(fontsize=8)
ax.set_ylim(0, max(3.5, ax.get_ylim()[1]))
ax.spines[["top", "right"]].set_visible(False)

# H4
ax = axes[2]
locations = list(H4_CONDITIONS.keys())
loc_labels = [f"agent{i}" for i in locations]
x4 = np.arange(len(locations))
for i, (model, mdict) in enumerate(models.items()):
    m4 = {"gpt-4o": h4_metrics, "gpt-4o-mini": h4_mini}[model]
    ss_vals = [m4.get(loc, {}).get("target_value_ss", float("nan")) for loc in locations]
    bars = ax.bar(x4 + i * width, ss_vals, width, label=model,
                  color=["#2a9d8f", "#8ecae6"][i], alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, ss_vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x4 + width / 2)
ax.set_xticklabels(loc_labels, fontsize=9)
ax.set_ylabel("SS(social_power)")
ax.set_title("H4: Location (chain)")
ax.legend(fontsize=8)
ax.set_ylim(0, max(3.5, ax.get_ylim()[1]))
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("gpt-4o vs gpt-4o-mini: System Susceptibility", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\nH2 Model Comparison — Mean SS across all conditions")
print(f"{'':20} {'gpt-4o':>10} {'gpt-4o-mini':>12} {'ratio':>8}")
print("-" * 52)

gpt4o_all = [m["target_value_ss"] for m in list(h1_metrics.values()) + list(h3_metrics.values()) + list(h4_metrics.values())]
mini_all  = [m["target_value_ss"] for m in list(h1_mini.values()) + list(h3_mini.values()) + list(h4_mini.values())]

mean4o = np.mean(gpt4o_all)
mean_mini = np.mean(mini_all)
print(f"{'Mean SS (all conds)':<20} {mean4o:>10.3f} {mean_mini:>12.3f} {mean4o/mean_mini if mean_mini else float('nan'):>8.2f}x")

# Per-sweep
for label, d4o, dmini in [
    ("H1 topology", h1_metrics, h1_mini),
    ("H3 value type", h3_metrics, h3_mini),
    ("H4 location", h4_metrics, h4_mini),
]:
    v4o = [m["target_value_ss"] for m in d4o.values()]
    vm  = [m["target_value_ss"] for m in dmini.values()]
    if v4o and vm:
        r = np.mean(v4o) / np.mean(vm) if np.mean(vm) else float("nan")
        print(f"{label:<20} {np.mean(v4o):>10.3f} {np.mean(vm):>12.3f} {r:>8.2f}x")

pred_supported = mean4o > mean_mini
print(f"\nH2 prediction (gpt-4o > gpt-4o-mini): {'✓ supported' if pred_supported else '✗ not supported'}")


## H5: Baseline Stability Check

**Statement:** Without perturbation, agent value scores remain stable across interaction rounds —
there is no systematic drift attributable to the discussion itself.

**Prediction:** Round-0 and final-round scores are statistically indistinguishable (|Δ| < 0.5
across all agents and values).

*(Paper Fig. 2 — stability check; score timeseries without perturbation)*

In [ ]:
if baseline_results is not None:
    agents = sorted(baseline_results.get_agents())
    values = baseline_results.get_values()

    # Compute per-agent per-value drift = final - initial
    drift_magnitudes = []
    drift_records = []
    for agent in agents:
        for value in values:
            scores = baseline_results.get_agent_value_scores(agent, value)
            if len(scores) >= 2:
                delta = abs(scores[-1] - scores[0])
                drift_magnitudes.append(delta)
                drift_records.append((agent, value, scores[0], scores[-1], scores[-1] - scores[0]))

    # Summary statistics
    drift_arr = np.array(drift_magnitudes)
    print(f"Baseline drift (|final - initial|) across {len(drift_arr)} agent-value pairs:")
    print(f"  Mean  : {drift_arr.mean():.3f}")
    print(f"  Max   : {drift_arr.max():.3f}")
    print(f"  Frac > 0.5: {(drift_arr > 0.5).mean():.1%}")

    threshold = 0.5
    if drift_arr.max() < threshold:
        print(f"\n✓ All drifts < {threshold}: baseline is stable. Prediction supported.")
    else:
        high = [(a, v, d, f, delta) for a, v, d, f, delta in drift_records if abs(delta) >= threshold]
        print(f"\n⚠  {len(high)} agent-value pairs drifted ≥ {threshold} — investigate:")
        for a, v, d, f, delta in sorted(high, key=lambda x: abs(x[4]), reverse=True)[:10]:
            print(f"   {a:<12} {v:<25} init={d:.1f} final={f:.1f} Δ={delta:+.1f}")

    # Plot score distribution across rounds for a few representative values
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    sample_values = [v for v in ["social_power", "helpful", "equality"] if v in values]
    if not sample_values:
        sample_values = values[:3]

    for ax, val in zip(axes, sample_values):
        for agent in agents:
            scores = baseline_results.get_agent_value_scores(agent, val)
            if scores:
                ax.plot(range(len(scores)), scores, marker="o", linewidth=1.5, label=agent)
        ax.set_title(val.replace("_", " "), fontsize=10)
        ax.set_xlabel("Round")
        ax.set_ylabel("Score (0–10)")
        ax.set_ylim(0, 10)
        ax.axhline(5, color="grey", linestyle=":", linewidth=0.8)
        ax.spines[["top", "right"]].set_visible(False)

    axes[-1].legend(title="Agent", fontsize=7, framealpha=0.7)
    fig.suptitle("Baseline Value Scores Over Rounds (no perturbation)", fontsize=12)
    fig.tight_layout()
    plt.show()

else:
    print("No baseline results yet.")
    print("Run: uv run python run_experiment.py scenario=valueflow evaluation=valueflow environment=valueflow scenario.perturbation.enabled=false")

## Summary: Cross-Hypothesis Comparison

Aggregate SS across all available conditions. Each bar represents the headline `target_value_ss` for that experimental condition.

In [ ]:
# Collect all SS values with labels
all_conditions: list[tuple[str, float]] = []

for topo, m in h1_metrics.items():
    all_conditions.append((f"[H1] {topo}", m["target_value_ss"]))

for vtype, m in h3_metrics.items():
    all_conditions.append((f"[H3] {vtype}", m["target_value_ss"]))

for idx, m in h4_metrics.items():
    label = {0: "head", 2: "middle", 4: "tail"}.get(idx, f"agent_{idx}")
    all_conditions.append((f"[H4] {label}", m["target_value_ss"]))

if all_conditions:
    labels_, ss_vals = zip(*sorted(all_conditions, key=lambda x: x[1], reverse=True))

    fig, ax = plt.subplots(figsize=(max(9, len(labels_) * 0.7), 5))
    colors = ["#e63946" if "[H1]" in l else "#457b9d" if "[H3]" in l else "#2a9d8f" for l in labels_]
    bars = ax.bar(labels_, ss_vals, color=colors, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, ss_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#e63946", label="H1: Topology"),
        Patch(facecolor="#457b9d", label="H3: Value type"),
        Patch(facecolor="#2a9d8f", label="H4: Location"),
    ]
    ax.legend(handles=legend_elements, fontsize=9)

    ax.set_title("System Susceptibility Across All Conditions", fontsize=13)
    ax.set_ylabel("SS (target_value_ss)")
    ax.set_xticklabels(labels_, rotation=35, ha="right", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()
else:
    print("No results loaded. Run the simulation pipeline first.")

In [ ]:
# Auto-evaluate hypothesis predictions where data is available
print("Hypothesis Evaluation Summary")
print("=" * 60)

# H1
if len(h1_metrics) == 4:
    ss_rank = sorted(h1_metrics, key=lambda t: h1_metrics[t]["target_value_ss"], reverse=True)
    predicted = ["fully_connected", "star", "ring", "chain"]
    match = ss_rank == predicted
    print(f"H1 topology  : {'✓ prediction matches' if match else '~ partial/reversed'}")
    print(f"   Observed order: {' > '.join(ss_rank)}")
    print(f"   Predicted order: {' > '.join(predicted)}")
elif h1_metrics:
    print(f"H1 topology  : {len(h1_metrics)}/4 conditions run")
else:
    print("H1 topology  : ○ no data")

print()

# H3
if len(h3_metrics) >= 4:
    ss_h3 = {vt: h3_metrics[vt]["target_value_ss"] for vt in h3_metrics}
    top2 = sorted(ss_h3, key=lambda x: ss_h3[x], reverse=True)[:2]
    predicted_top = {"power", "achievement"}
    print(f"H3 value type: {'✓ self-enhancing top' if set(top2) == predicted_top else '~ does not match'} (top 2: {top2})")
elif h3_metrics:
    print(f"H3 value type: {len(h3_metrics)}/6 conditions run")
else:
    print("H3 value type: ○ no data")

print()

# H4
if len(h4_metrics) == 3:
    head, mid, tail = h4_metrics.get(0, {}).get("target_value_ss", None), \
                      h4_metrics.get(2, {}).get("target_value_ss", None), \
                      h4_metrics.get(4, {}).get("target_value_ss", None)
    if all(v is not None for v in [head, mid, tail]):
        monotonic = head >= mid >= tail
        print(f"H4 location  : {'✓ monotonic decrease' if monotonic else '✗ non-monotonic'}")
        print(f"   head={head:.2f}  middle={mid:.2f}  tail={tail:.2f}")
elif h4_metrics:
    print(f"H4 location  : {len(h4_metrics)}/3 conditions run")
else:
    print("H4 location  : ○ no data")

print()

# H5
if baseline_results is not None and drift_magnitudes:
    frac_over = (np.array(drift_magnitudes) > 0.5).mean()
    print(f"H5 baseline  : {'✓ stable (max drift < 0.5)' if max(drift_magnitudes) < 0.5 else f'⚠ {frac_over:.1%} of agent-value pairs drift > 0.5'}")
else:
    print("H5 baseline  : ○ no data")

## Takeaways

*(Fill in after running experiments — placeholder structure below)*

### Key Findings

- **H1 topology:** [e.g., SS follows predicted order: fully_connected > star > ring > chain, confirming that denser connection structures amplify value propagation]
- **H3 value type:** [e.g., self-enhancing values (power, achievement) show significantly higher SS than conservation values, matching Schwartz theory predictions]
- **H4 location:** [e.g., head perturbation (agent_0) produces {X}x higher SS than tail perturbation (agent_4), confirming DAG distance attenuates influence]
- **H5 baseline:** [e.g., Without perturbation, all 56 Schwartz values remain within ±0.3 across all agents and rounds — baseline is stable]
- **β-spillover:** [e.g., Perturbing `social_power` also lifts `authority` and `wealth` (Pearson r ≈ 0.6), but does not affect conservation values (r < 0.1)]

### Limitations

- **n=1 run per condition** — no seed replication, so statistical significance cannot be assessed. Replicate with ≥3 seeds before drawing strong conclusions.
- **5-agent networks only** — propagation dynamics may differ in larger or heterogeneous networks.
- **Single model (GPT-4o) for H1/H3/H4** — H2 (model backbone) is planned but not yet run.
- **Probe validity** — direct self-report of value endorsement may be biased by social desirability even for LLMs.

### Next Steps

- Run H2 (model backbone): compare GPT-4o vs GPT-4o-mini vs Claude on chain topology.
- Add seed replication (n=3) to compute confidence intervals on SS.
- Extend to larger networks (10-agent) to test scaling behaviour of SS with network size.
- Try the judged probe variant (`evaluation=valueflow_judged`) and compare SS estimates to direct-report.